# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_obj = dataset.metadata
metadata_dict = metadata_obj.to_json()
print("Dataset Title:", metadata_obj.name)
print("Description:", metadata_obj.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets (by @id) and their fields
record_set_ids = []
record_sets_info = []
print("Available record sets and fields:")
for record_set in metadata_obj.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    record_set_ids.append(record_set.id)
    fields = getattr(record_set, 'fields', [])
    for field in fields:
        print(f"    Field @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
        record_sets_info.append({'record_set_id': record_set.id, 'field_id': field.id, 'field_name': field.name})

## 2.1 Preview Records for Each RecordSet
Let's preview the first 2 records in each available record set.

In [ ]:
# Show first 2 records for each record set
for rs_id in record_set_ids:
    print(f"\nSample records from RecordSet @id: {rs_id}")
    try:
        for idx, record in enumerate(dataset.records(record_set=rs_id)):
            print(json.dumps(record, indent=2))
            if idx >= 1:
                break
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We reference each entity by its `@id`.

In [ ]:
# Extract data from all available record sets
dataframes = {}
for record_set_id in record_set_ids:
    # Attempt to load all records into a DataFrame
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id: {record_set_id}")
        else:
            print(f"No records found in RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# For demonstration, select the largest DataFrame as our main data for EDA
if len(dataframes):
    # Choose the record set with the most records
    main_record_set_id = max(dataframes.keys(), key=lambda k: len(dataframes[k]))
    print(f"\nMain RecordSet for analysis: {main_record_set_id}")
    print("Columns in this DataFrame:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# We'll attempt EDA if there's a main data frame
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    # Guess a numeric field: select the first column with numeric dtype or typical mass spec names
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try to coerce commonly named columns
        for possible in ['MW', 'molecular_weight', 'coverage', 'Abundance', 'Normalized abundance']:
            if possible in df.columns:
                try:
                    df[possible] = pd.to_numeric(df[possible], errors='coerce')
                    if pd.api.types.is_numeric_dtype(df[possible]):
                        numeric_candidates.append(possible)
                        break
                except Exception:
                    continue

    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # First numeric field
        print(f"Chosen numeric field for demo: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as example cutoff

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (approx. top 25%)")
        display(filtered_df.head())

        # Normalize that field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a common categorical field for proteins
        possible_group_fields = [col for col in df.columns if col.lower() in ['description', 'accession', 'sample', 'modification', 'group']]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"\nGrouping by '{group_field}': show mean normalized values per group (first 5)")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name='mean_'+numeric_field)
            display(grouped_df.head())
        else:
            print("No categorical group field found to group by.")
    else:
        print("No obvious numeric field found for EDA in the main record set.")
else:
    print("No main record set loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_candidates:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of '{numeric_field}'")
    plt.show()

    # If group_field exists and has few unique values, boxplot
    if 'group_field' in locals():
        unique_count = df[group_field].nunique()
        if unique_count < 30:
            plt.figure(figsize=(12, 5))
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f"'{numeric_field}' by '{group_field}'")
            plt.xticks(rotation=40)
            plt.show()
else:
    print("Insufficient numeric data for visualization.")

## 6. Conclusion
This notebook demonstrates how to use `mlcroissant` for loading and exploring a dataset defined by a Croissant schema.

**Key findings:**
- Loaded metadata and records using the Croissant dataset URL.
- Identified available record sets, their fields and corresponding `@id`s.
- Extracted and previewed records using their `@id`s.
- Performed basic EDA: filtered on a numeric field, normalized values, and grouped by protein/description as available.
- Visualized field distributions to identify potential data structure for analysis.

For more advanced processing, you can refer to dataset-specific documentation or explore additional record sets/fields using their `@id`s as shown above.